# Data Preprocessing

The purpose of this notebook is to handle all the preprocessing. Data is exported to `../data/notebooks_exports/`.

_structure work in progress_

## libraries

In [ ]:
import json
import random
import pandas as pd
import numpy as np

from pathlib import Path

## settings

In [ ]:
DATA_DIR = Path("../data")
EXPORT_DIR = DATA_DIR / "notebooks_exports"

INPUT_DATA = DATA_DIR / "thesis_data_1778418853489.json"

TICKS_PER_SECOND = 20 # amount of ticks (steps) per second in minecraft

MIN_WINDOW_SIZE = 60 #size for windows to be considered windows
RANDOM_WINDOW_SIZE = 60 # size for random windows in ticks
RANDOM_SEED = 18 # to make the random windows reproducable

IDLE_THRESHOLD = 0.01 # distance moved that is still considered no movement
MAX_IDLE_TICKS = 3 # consecutive ticks without movement needed for idle
MIN_BLOCKS = 5 # min amount of blocks in window to make data meaningful

CONCEPT_WINDOW_SIZE = 50 # window size in ticks for mcplayhd's approach
Y_WEIGHT = 0.5 # weight of the y distance when searching for the best window
MAX_CLICKS = 5 # clicks per tick get capped at this value
EYE_HEIGHT = 1.62 # height of the player's eyes above their position

## importing data

In [ ]:
# load the json
with open(INPUT_DATA, "r", encoding="utf-8") as f:
    data = json.load(f)

# sanity checks
print(f"data set: {len(data)} replays")
print(f"replay structure: {list(data[0].keys())}")

## 01 - blocks / clicks & movement

#### _helper functions_

In [ ]:
def movement_per_tick(steps):
    """moved distance between each tick, returns an (x-1, ) 1D array, where x is the number of input ticks"""
    positions = np.array([[ # (x, 3) array with the (x,y,z) positions
        step["playerPosition"]["x"],
        step["playerPosition"]["y"],
        step["playerPosition"]["z"]
    ] for step in steps])
    deltas = np.diff(positions, axis=0) # (x-1, 3) array with the differences between each row
    movement = np.sqrt((deltas ** 2).sum(axis=1)) # (x-1, ) 1D array with calculated distance in each row

    return movement


def longest_active_window(movement):
    """finds the longest active window in movement, returns a slice or none if too short"""
    is_idle = movement < IDLE_THRESHOLD
    keep = np.ones(len(movement), dtype=bool)
    idle_ticks_combo = 0

    for i in range(len(movement)):
        if is_idle[i]:
            idle_ticks_combo += 1
        else:
            idle_ticks_combo = 0

        if idle_ticks_combo > MAX_IDLE_TICKS:
            # drop the whole idle part
            keep[i - idle_ticks_combo + 1 : i+1] = False

    best_start, best_length = 0, 0
    current_start, current_length = 0, 0

    for i in range(len(keep)):
        if keep[i]:
            # if keep and no active window, we start new window
            if current_length == 0:
                current_start = i
            current_length += 1
            # if current window is better, we put it as best window
            if current_length > best_length:
                best_start, best_length = current_start, current_length #
        else:
            # if not keep, we drop the window by setting length to 0
            current_length = 0

    if best_length < MIN_WINDOW_SIZE:
        return None

    return slice(best_start, best_start + best_length)


def replay_features(item):
    """calculates blocks/tick and average movement speed in replay, returns as dict"""
    # drop replays that are too short
    steps = item["stepData"]
    if len(steps) < MIN_WINDOW_SIZE:
        return None

    # drop replays with no viable window
    movement = movement_per_tick(steps)
    window = longest_active_window(movement)
    if window is None:
        return None

    # count clicks/block in window
    # use [1:] because movement of ticks 0-1 belong to step 1
    clicks = np.array([step["clicks"] for step in steps[1:]])[window].sum()
    blocks = np.array([len(step["blocksChanged"]) for step in steps[1:]])[window].sum()

    # drop if too few blocks placed in window
    if blocks < MIN_BLOCKS or clicks == 0:
        return None

    return {
        "fastBuilderMode": item["fastBuilderMode"],
        "idAnonymous": item["idAnonymous"],
        "blocks_per_click": blocks / clicks,
        "avg_speed": movement[window].mean() * TICKS_PER_SECOND,
        "window_share": len(movement[window]) / len(movement),
    }

#### building dataframe from whole dataset

In [ ]:
rows = []
for item in data:
    row = replay_features(item)
    if row is not None:
        rows.append(row)

df_speed_to_block_per_click = pd.DataFrame(rows)

# sanity checks
print(f"dataframe holds {len(df_speed_to_block_per_click)} out of {len(data)} replays")
print(df_speed_to_block_per_click.head(3))


#### exporting the dataframe

In [ ]:
EXPORT_DIR.mkdir(exist_ok=True)
df_speed_to_block_per_click.to_csv(EXPORT_DIR / "speed_to_block_per_click.csv", index=False)

## 02- random windows (wide dataframe)

#### _helper functions_

In [ ]:
def random_window(steps):
    """create a random window within replays, returns a list of steps"""
    # drop if too short to hold a valid window
    if len(steps) < RANDOM_WINDOW_SIZE:
        return None

    # drop if no valid active window
    active = longest_active_window(movement_per_tick(steps))
    if active is None:
        return None
    active_steps = steps[1:][active] # move active steps by 1 to align with movement

    # drop if active window too small
    if len(active_steps) < RANDOM_WINDOW_SIZE:
        return None
    start = random.randint(0, len(active_steps) - RANDOM_WINDOW_SIZE)
    return active_steps[start : start + RANDOM_WINDOW_SIZE]


def window_features(item, window):
    """creates the wide data rows, returns as dict"""
    movement = movement_per_tick(window)
    yaw = np.array([step["playerPosition"]["yaw"] for step in window])
    pitch = np.array([step["playerPosition"]["pitch"] for step in window])
    ping = np.array([step["ping"] for step in window])
    yaw_change = (np.diff(yaw) + 180) % 360 - 180 # yaw isn't limited to -180 to 180, so we limit it
    pitch_change = np.diff(pitch)

    # create the columns
    row = {
        "idAnonymous": item["idAnonymous"],
        "fastBuilderMode": item["fastBuilderMode"],
        "pingAvg": ping.mean(), # average ping
        "pingJitter": np.abs(np.diff(ping)).mean(), # average ping change between ticks
    }

    # fill data in rows
    # creates elements like movement_1, movement_2 and so on
    row.update({f"movement_{i}": value for i, value in enumerate(movement, start=1)})
    row.update({f"yaw_change_{i}": value for i, value in enumerate(yaw_change, start=1)})
    row.update({f"pitchChange_{i}": value for i, value in enumerate(pitch_change, start=1)})
    row.update({f"clicks_{i}": step["clicks"] for i, step in enumerate(window)})
    row.update({f"sneaking_{i}": int(step["sneaking"]) for i, step in enumerate(window)})
    row.update({f"blocksChanged_{i}": len(step["blocksChanged"]) for i, step in enumerate(window)})

    return row



#### create dataframe

In [ ]:
random.seed(RANDOM_SEED)

rows = []
for item in data:
    window = random_window(item["stepData"])
    if window is not None:
        rows.append(window_features(item, window))

df_wide = pd.DataFrame(rows)

# sanity checks
print(f"dataframe holds {len(df_wide)} out of {len(data)} replays")
# preview: id/ping columns plus only ticks 1-3 of each per-tick feature (df_wide itself keeps all 361 columns)
print(df_wide.filter(regex=r"^(idAnonymous|fastBuilderMode|pingAvg|pingJitter)$|_[1-3]$").head(3))

#### export data

In [ ]:
EXPORT_DIR.mkdir(exist_ok=True)
df_wide.to_csv(EXPORT_DIR / "wide_table.csv", index=False)

## 03 - McPlayHD suggestion (custom wide dataframe)

#### _helper functions_

In [ ]:
def last_block_tick(steps):
    """index of the last step that placed a non-AIR block, returns an int or None if no block was placed"""
    tick_block_changed = None
    for i, step in enumerate(steps):
        for block_changed in step["blocksChanged"]:
            if block_changed["material"] != "AIR":
                tick_block_changed = i

    return tick_block_changed


def best_window_start(steps, last_tick):
    """finds the start of the window that covers the most distance without going down, returns an int or None if no window fits"""
    if last_tick is None:
        return None

    best_start = None
    best_distance = -1

    for i, step in enumerate(steps):
        # exit loop if too few ticks for window size
        if last_tick - i < CONCEPT_WINDOW_SIZE:
            break

        player_start_position = step["playerPosition"]
        player_end_position = steps[i + CONCEPT_WINDOW_SIZE - 1]["playerPosition"]

        # skip window if end position below start position
        if player_start_position["y"] > player_end_position["y"]:
            continue

        dx = player_start_position["x"] - player_end_position["x"]
        dy = player_start_position["y"] - player_end_position["y"]
        dz = player_start_position["z"] - player_end_position["z"]
        weighted_distance_squared = dx ** 2 + (dy * Y_WEIGHT) ** 2 + dz ** 2

        # check window and save it if it is better
        if weighted_distance_squared > best_distance:
            best_start = i
            best_distance = weighted_distance_squared

    return best_start


def blocks_before(steps, start):
    """positions of all blocks placed before the window, returns a set of (x, y, z) tuples"""
    block_set = set()
    for step in steps[:start]:
        for block_changed in step["blocksChanged"]:
            if block_changed["material"] != "AIR":
                location = block_changed["loc"]
                block_set.add((location["x"], location["y"], location["z"]))

    return block_set


def concept_window_features(item, start, already_placed):
    """creates the wide data rows for the window starting at start, returns as dict"""
    steps = item["stepData"]
    window = steps[start : start + CONCEPT_WINDOW_SIZE]

    x = np.array([step["playerPosition"]["x"] for step in window])
    dx = np.diff(x)
    y = np.array([step["playerPosition"]["y"] for step in window])
    dy = np.diff(y)
    z = np.array([step["playerPosition"]["z"] for step in window])
    dz = np.diff(z)
    yaw = np.array([step["playerPosition"]["yaw"] for step in window])
    yaw_change = (np.diff(yaw) + 180) % 360 - 180
    pitch = np.array([step["playerPosition"]["pitch"] for step in window])
    pitch_change = np.diff(pitch)

    row = {
        "idAnonymous": item["idAnonymous"],
        "fastBuilderMode": item["fastBuilderMode"],
    }

    row.update({f"dx_{i}": value for i, value in enumerate(dx, start=1)})
    row.update({f"dy_{i}": value for i, value in enumerate(dy, start=1)})
    row.update({f"dz_{i}": value for i, value in enumerate(dz, start=1)})
    row.update({f"yaw_change_{i}": value for i, value in enumerate(yaw_change, start=1)})
    row.update({f"pitch_change_{i}": value for i, value in enumerate(pitch_change, start=1)})
    row.update({f"sneaking_{i}": int(step["sneaking"]) for i, step in enumerate(window)})
    row.update({f"clicks_{i}": min(step["clicks"], MAX_CLICKS) for i, step in enumerate(window)})

    for i, step in enumerate(window):
        new_blocks = []
        for block_changed in step["blocksChanged"]:
            loc = block_changed["loc"]
            position = (loc["x"], loc["y"], loc["z"])
            if (block_changed["material"] != "AIR") and (position not in already_placed):
                new_blocks.append(position)

    return row

In [ ]:
# temp cell
item = data[0]
steps = item["stepData"]

start = best_window_start(steps, last_block_tick(steps))
row = concept_window_features(item, start, blocks_before(steps, start))
print(len(row))
print(list(row)[:5])



#### create dataframe

In [ ]:
# TODO: loop over data, collect one row per replay, build df_concept

#### export data

In [ ]:
# TODO: export df_concept as csv